<a href="https://colab.research.google.com/github/fatimasood/nlp-text-preprocessing-imdb/blob/main/IMDB_Dataset_of_50K_Movie_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOAD AND INSTALL LIBRARIES

In [11]:
!pip install emoji --quiet
!pip install contractions --quiet
!pip install pyspellchecker --quiet
import emoji
import pandas as pd
import re
import string
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import contractions
from spellchecker import SpellChecker
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 55.4 MB/s eta 0:00:00


DATASET...

In [12]:
# Load the dataset
df = pd.read_csv('/content/sample_data/IMDB Dataset.csv', engine='python', quotechar='"', escapechar='\\', on_bad_lines='skip')
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Sentiment Distribution:\n{df['sentiment'].value_counts()}")
print(f"\n--- Sample Raw Reviews ---")
for i in range(3):
    print(f"\nReview {i+1} [{df['sentiment'].iloc[i]}]:")
    print(df['review'].iloc[i][:300], "...")

Dataset Shape: (50000, 2)
Columns: ['review', 'sentiment']
Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

--- Sample Raw Reviews ---

Review 1 [positive]:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru ...

Review 2 [positive]:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari"  ...

Review 3 [positive]:
I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hea

Dataset Statistics

In [13]:
# Compute key statistics
review_lengths = df['review'].apply(lambda x: len(x.split()))
print(f"\n--- Review Length Statistics (in words) ---")
print(f"Mean: {review_lengths.mean():.1f}")
print(f"Median: {review_lengths.median():.0f}")
print(f"Max: {review_lengths.max()}")
print(f"Min: {review_lengths.min()}")

# Count HTML tags
html_count = df['review'].apply(lambda x: '<br' in x).sum()
print(f"\nReviews containing HTML tags: {html_count} ({html_count/len(df)*100:.1f}%)")

# Count emojis
emoji_count = df['review'].apply(lambda x: emoji.emoji_count(x)).sum()
print(f"Total emojis in dataset: {emoji_count}")


--- Review Length Statistics (in words) ---
Mean: 231.2
Median: 173
Max: 2470
Min: 4

Reviews containing HTML tags: 29200 (58.4%)
Total emojis in dataset: 5


Preprocessing Pipeline


In [14]:
def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to first 1000 reviews for demonstration (scalable to full dataset)
sample_reviews = df['review'].head(1000).tolist()
cleaned_corpus = [clean_text(review) for review in sample_reviews]
print("--- Cleaned Sample (First Review) ---")
print(cleaned_corpus[0][:500])

--- Cleaned Sample (First Review) ---
one of the other reviewers has mentioned that after watching just oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the wordit is called oz as that is the nickname given to the 


Tokenization

In [15]:
nltk.download('punkt')
nltk.download('punkt_tab')

tokenized_corpus = [word_tokenize(doc) for doc in cleaned_corpus]
print("--- Tokenized (First Review, First 30 Tokens) ---")
print(tokenized_corpus[0][:30])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- Tokenized (First Review, First 30 Tokens) ---
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'methe', 'first', 'thing', 'that']


Stopword Removal

In [16]:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
filtered_corpus = [[word for word in doc if word not in stop_words] for doc in tokenized_corpus]

print("--- Stopwords Removed (First Review, First 20 Tokens) ---")
print(filtered_corpus[0][:20])
print(f"\nOriginal token count (avg): {sum(len(doc) for doc in tokenized_corpus)/len(tokenized_corpus):.0f}")
print(f"After stopword removal (avg): {sum(len(doc) for doc in filtered_corpus)/len(filtered_corpus):.0f}")

--- Stopwords Removed (First Review, First 20 Tokens) ---
['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence']

Original token count (avg): 225
After stopword removal (avg): 119


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Stemming vs. Lemmatization

In [17]:
nltk.download('wordnet')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_corpus = [[stemmer.stem(word) for word in doc] for doc in filtered_corpus]
lemmatized_corpus = [[lemmatizer.lemmatize(word) for word in doc] for doc in filtered_corpus]

# Compare on first review
comparison = pd.DataFrame({
    'Original': filtered_corpus[0][10:25],
    'Stemmed': stemmed_corpus[0][10:25],
    'Lemmatized': lemmatized_corpus[0][10:25]
})
print("\n--- Stemming vs Lemmatization Comparison ---")
print(comparison.to_string(index=False))

[nltk_data] Downloading package wordnet to /root/nltk_data...



--- Stemming vs Lemmatization Comparison ---
   Original  Stemmed  Lemmatized
   happened   happen    happened
      methe     meth       methe
      first    first       first
      thing    thing       thing
     struck   struck      struck
         oz       oz          oz
  brutality   brutal   brutality
unflinching unflinch unflinching
     scenes    scene       scene
   violence  violenc    violence
        set      set         set
      right    right       right
       word     word        word
         go       go          go
      trust    trust       trust


Contractions Expansion

In [18]:
# Apply on first 1000 reviews
expanded_sample = [contractions.fix(review) for review in sample_reviews[:5]]
print("--- Contractions Expanded (First 2 Reviews, Excerpts) ---")
for i in range(2):
    print(f"\nOriginal: {sample_reviews[i][:150]}...")
    print(f"Expanded: {expanded_sample[i][:150]}...")

--- Contractions Expanded (First 2 Reviews, Excerpts) ---

Original: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with...
Expanded: One of the other reviewers has mentioned that after watching just 1 Oz episode you will be hooked. They are right, as this is exactly what happened wi...

Original: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes d...
Expanded: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes d...


Emoji Detection and Conversion

In [19]:
# Find reviews containing emojis
emoji_reviews = df['review'].apply(lambda x: emoji.emoji_count(x) > 0)
print(f"Reviews with emojis: {emoji_reviews.sum()} ({emoji_reviews.sum()/len(df)*100:.2f}%)")

# Demo on reviews with emojis
emoji_examples = df[emoji_reviews]['review'].head(3).tolist()
for i, review in enumerate(emoji_examples):
    converted = emoji.demojize(review)
    print(f"\n--- Emoji Review {i+1} ---")
    print(f"Original: ...{review[review.index('☺'):review.index('☺')+50] if '☺' in review else review[:100]}...")
    print(f"Converted: ...{converted[converted.index(':smiling'):converted.index(':smiling')+50] if ':smiling' in converted else converted[:100]}...")

Reviews with emojis: 5 (0.01%)

--- Emoji Review 1 ---
Original: ...I checked this movie out based on a favorable review on this page. It is slow moving and the payoff ...
Converted: ...I checked this movie out based on a favorable review on this page. It is slow moving and the payoff ...

--- Emoji Review 2 ---
Original: ..."In April 1946, the University of Chicago agreed to operate Argonne National Laboratory, with an ass...
Converted: ..."In April 1946, the University of Chicago agreed to operate Argonne National Laboratory, with an ass...

--- Emoji Review 3 ---
Original: ...That's the sound of Stan and Ollie spinning in their graves.<br /><br />I won't bother listing the f...
Converted: ...That's the sound of Stan and Ollie spinning in their graves.<br /><br />I won't bother listing the f...


Spell Correction

In [20]:
spell = SpellChecker()

# Apply to a small batch (spell checking is computationally expensive)
test_tokens = tokenized_corpus[0][:30]
corrected_tokens = [spell.correction(word) if spell.correction(word) is not None else word for word in test_tokens]

print("--- Spell Correction (First Review, First 30 Tokens) ---")
for orig, corr in zip(test_tokens, corrected_tokens):
    if orig != corr:
        print(f"  {orig:15s} → {corr}")

--- Spell Correction (First Review, First 30 Tokens) ---
  youll           → you'll
  methe           → mete


POS Tagging

In [22]:
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets_json')

pos_tagged = [nltk.pos_tag(doc) for doc in tokenized_corpus]
print("--- POS Tagging (First Review, First 15 Tags) ---")
for word, tag in pos_tagged[0][:15]:
    print(f"  {word:15s} → {tag:6s} ({nltk.help.upenn_tagset(tag)})")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets_json to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets_json.zip.


--- POS Tagging (First Review, First 15 Tags) ---
CD: numeral, cardinal
    mid-1890 nine-thirty forty-two one-tenth ten million 0.5 one forty-
    seven 1987 twenty '79 zero two 78-degrees eighty-four IX '60s .025
    fifteen 271,124 dozen quintillion DM2,000 ...
  one             → CD     (None)
IN: preposition or conjunction, subordinating
    astride among uppon whether out inside pro despite on by throughout
    below within for towards near behind atop around if like until below
    next into if beside ...
  of              → IN     (None)
DT: determiner
    all an another any both del each either every half la many much nary
    neither no some such that the them these this those
  the             → DT     (None)
JJ: adjective or numeral, ordinal
    third ill-mannered pre-war regrettable oiled calamitous first separable
    ectoplasmic battery-powered participatory fourth still-to-be-named
    multilingual multi-disciplinary ...
  other           → JJ     (None)
NNS: noun, comm

In [23]:
def full_preprocessing_pipeline(text, remove_stopwords=True, use_lemmatization=True):
    """
    Complete preprocessing pipeline for IMDb reviews.
    """
    # Contractions expansion
    text = contractions.fix(text)
    # Emoji conversion
    text = emoji.demojize(text)
    # Lowercase
    text = text.lower()
    # HTML removal
    text = BeautifulSoup(text, "html.parser").get_text()
    # URL removal
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Number removal
    text = re.sub(r'\d+', '', text)
    # Punctuation removal
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenization
    tokens = word_tokenize(text)
    # Optional stopword removal
    if remove_stopwords:
        tokens = [w for w in tokens if w not in stop_words]
    # Normalization (stemming or lemmatization)
    if use_lemmatization:
        tokens = [lemmatizer.lemmatize(w) for w in tokens]
    else:
        tokens = [stemmer.stem(w) for w in tokens]
    return tokens

# Test pipeline
test_review = df['review'].iloc[42]  # Random review
processed = full_preprocessing_pipeline(test_review)
print(f"Original length: {len(test_review.split())} words")
print(f"Processed length: {len(processed)} tokens")
print(f"Sample tokens: {processed[:25]}")

Original length: 176 words
Processed length: 89 tokens
Sample tokens: ['film', 'seen', 'one', 'rage', 'got', 'one', 'worst', 'yet', 'direction', 'logic', 'continuity', 'change', 'plotscript', 'dialog', 'made', 'cry', 'pain', 'could', 'anyone', 'come', 'something', 'crappy', 'gary', 'busey', 'know']


Vocabulary Analysis: Before vs. After Preprocessing

In [24]:
# Before preprocessing
raw_vocab = set()
for review in sample_reviews[:500]:
    raw_vocab.update(review.lower().split())

# After preprocessing
processed_vocab = set()
for tokens in lemmatized_corpus[:500]:
    processed_vocab.update(tokens)

print(f"--- Vocabulary Reduction ---")
print(f"Raw vocabulary size (500 reviews): {len(raw_vocab):,}")
print(f"Processed vocabulary size: {len(processed_vocab):,}")
print(f"Reduction: {(1 - len(processed_vocab)/len(raw_vocab))*100:.1f}%")

--- Vocabulary Reduction ---
Raw vocabulary size (500 reviews): 20,692
Processed vocabulary size: 12,856
Reduction: 37.9%


Impact on Downstream Task: Sentiment Classification

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

# Prepare data
sample_df = df.head(2000)
X_raw = sample_df['review']
X_processed = [' '.join(full_preprocessing_pipeline(r)) for r in sample_df['review']]
y = (sample_df['sentiment'] == 'positive').astype(int)

# TF-IDF Vectorization
vec_raw = TfidfVectorizer(max_features=5000)
vec_proc = TfidfVectorizer(max_features=5000)

X_raw_vec = vec_raw.fit_transform(X_raw)
X_proc_vec = vec_proc.fit_transform(X_processed)

# Classification
clf = LogisticRegression(max_iter=1000)
score_raw = cross_val_score(clf, X_raw_vec, y, cv=5).mean()
score_proc = cross_val_score(clf, X_proc_vec, y, cv=5).mean()

print(f"\n--- Classification Performance (5-fold CV) ---")
print(f"Accuracy with RAW text:    {score_raw:.4f}")
print(f"Accuracy with PROCESSED text: {score_proc:.4f}")
print(f"Improvement: {((score_proc - score_raw)/score_raw)*100:.2f}%")


--- Classification Performance (5-fold CV) ---
Accuracy with RAW text:    0.8350
Accuracy with PROCESSED text: 0.8390
Improvement: 0.48%
